# 04 (OPTIONAL, Colab) - Fine-tune DistilBERT on the category labels

**Run this in Google Colab with a GPU** (Runtime > Change > T4 GPU). It is the one
step that is impractical on an 8GB CPU laptop, so it is kept separate.

Upload `data/processed/reviews_clustered.csv` (or connect Drive) and the model
comparison in the README can be completed with a transformer baseline.

In [1]:
# @title Setup
# Colab-only guard: run locally and the !pip line below would install packages
# into the project venv, and the GPU assert would then fail confusingly.
try:
    from google.colab import files  # noqa: F401
except ImportError:
    raise SystemExit(
        '04 is Colab-only. Open it in Google Colab with a T4 GPU '
        '(Runtime > Change runtime type). Running it here would also '
        'pip-install into the local venv.')
!pip -q install transformers datasets scikit-learn accelerate
import os, numpy as np, pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Switch the Colab runtime to a GPU!'
print('device:', torch.cuda.get_device_name(0))

SystemExit: 04 is Colab-only. Open it in Google Colab with a T4 GPU (Runtime > Change runtime type). Running it here would also pip-install into the local venv.

/home/sudowoodo/Pro/review-organizer/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# @title Load data (upload reviews_clustered.csv to this Colab session)
from google.colab import files
if not os.path.exists('reviews_clustered.csv'):
    files.upload()
df = pd.read_csv('reviews_clustered.csv')
# Must stay in sync with config/cluster_names.json: 03 trains on these exact
# strings, so a divergent list makes the transformer-vs-classical comparison
# in the README meaningless.
NAMES = {'0': "app crashes & won't open", '1': 'mixed / general impressions',
         '2': 'reminders & notifications broken', '3': 'pricing & ads complaints',
         '4': 'task & scheduling feature requests',
         '5': 'calendar & event integration', '6': 'usability & ease of use',
         '7': 'positive feedback / praise'}
unmapped = sorted(set(df['cluster'].astype(str)) - set(NAMES))
assert not unmapped, ('clusters %s are not in NAMES - sync it with '
                      'config/cluster_names.json' % unmapped)
df['label'] = df['cluster'].astype(str).map(NAMES)
labels = sorted(df['label'].unique())
label2id = {l:i for i,l in enumerate(labels)}
df['label_id'] = df['label'].map(label2id)
print(len(df), 'rows,', len(labels), 'classes')

In [ ]:
# Stratified split mirroring the classical models (70/15/15)
tr, tmp = train_test_split(df, test_size=0.3, stratify=df['label_id'], random_state=SEED)
va, te = train_test_split(tmp, test_size=0.5, stratify=tmp['label_id'], random_state=SEED)
print(len(tr), len(va), len(te))

In [ ]:
# @tokenize into HF datasets
from datasets import Dataset
from transformers import AutoTokenizer
MODEL = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(MODEL)
def to_hf(d):
    return Dataset.from_dict({'text': list(d['raw_text'].astype(str)),
                              'label': list(d['label_id'])})
def mtl(ex):
    return tok(ex['text'], truncation=True, max_length=128, padding='max_length')
ds = {k: v.map(mtl, batched=True) for k, v in
      {'train': to_hf(tr), 'val': to_hf(va), 'test': to_hf(te)}.items()}

In [ ]:
# @title Fine-tune
from transformers import (AutoModelForSequenceClassification, TrainingArguments, Trainer,
                            DataCollatorWithPadding)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(labels))
args = TrainingArguments(
    output_dir='distilbert-reviews',
    learning_rate=2e-5, per_device_train_batch_size=16,
    per_device_eval_batch_size=32, num_train_epochs=3,
    weight_decay=0.01, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='f1_macro',
    seed=SEED, fp16=torch.cuda.is_available(), report_to=[])
def metrics(eval_pred):
    logits, y = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(y, pred),
            'f1_macro': f1_score(y, pred, average='macro')}
# transformers >=5 dropped the `tokenizer=` alias for `processing_class=`
trainer = Trainer(model=model, args=args, train_dataset=ds['train'],
                  eval_dataset=ds['val'], processing_class=tok,
                  compute_metrics=metrics,
                  data_collator=DataCollatorWithPadding(tok))
trainer.train()

In [ ]:
# @title Test-set report + compare against classical models in README
preds = trainer.predict(ds['test'])
yhat = np.argmax(preds.predictions, axis=-1)
ytrue = np.array(ds['test']['label'])
print(classification_report(ytrue, yhat, target_names=labels, zero_division=0))
print('DistilBERT accuracy:', round(accuracy_score(ytrue, yhat), 4),
      '| macro-F1:', round(f1_score(ytrue, yhat, average='macro'), 4))
trainer.save_model('distilbert-reviews-final')
# Download for the README results table:
!zip -rq distilbert-reviews-final.zip distilbert-reviews-final
from google.colab import files; files.download('distilbert-reviews-final.zip')

Paste the DistilBERT accuracy / macro-F1 into the README results table and compare
with the classical TF-IDF models. DistilBERT usually wins on paraphrase-heavy and
sarcasm-heavy categories but costs GPU time and is slower to serve.